# FloodLLM Validation Notebook

This notebook validates the flood detection pipeline against known flood events.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## 1. Test Data Setup

In [ ]:
# Create synthetic flood data for testing
def create_synthetic_flood_mask(size=100, flood_fraction=0.2):
    """Create synthetic flood mask for testing."""
    mask = np.zeros((size, size), dtype=bool)
    
    # Simulate river flooding (flood spreads from center)
    center = size // 2
    y, x = np.ogrid[:size, :size]
    distance = np.sqrt((x - center)**2 + **(y - center)2)
    
    # Flood in river valley pattern
    flood_threshold = np.percentile(distance, flood_fraction * 100)
    mask[distance < flood_threshold] = True
    
    return mask

# Create test masks
flood_mask = create_synthetic_flood_mask(flood_fraction=0.2)
print(f"Synthetic flood mask created: {np.sum(flood_mask)} flooded pixels")

## 2. SAR Processing Validation

In [ ]:
from app.processing.sar_processor import SARProcessor

processor = SARProcessor()

# Test threshold calculation
vv_data = np.random.normal(-15, 8, (100, 100)).astype(np.float32)
vv_data[flood_mask] = np.random.normal(-22, 3, np.sum(flood_mask))

threshold = processor._calculate_otsu_threshold(vv_data)
print(f"Otsu threshold: {threshold:.2f} dB")

# Apply threshold
detected_water = vv_data < threshold

# Calculate accuracy
iou = np.sum(flood_mask & detected_water) / np.sum(flood_mask | detected_water)
print(f"IoU with ground truth: {iou:.3f}")

## 3. NDWI Validation

In [ ]:
from app.processing.optical import OpticalProcessor

optical = OpticalProcessor()

# Create synthetic Sentinel-2 bands
# Water: high green (0.3), low NIR (0.1)
# Land: lower green (0.15), higher NIR (0.35)
green = np.where(flood_mask, 0.3, 0.15).astype(np.float32)
nir = np.where(flood_mask, 0.1, 0.35).astype(np.float32)

# Calculate NDWI
ndwi = optical._compute_ndwi(green, nir)

# Threshold NDWI
ndwi_water = ndwi > 0.2

# Compare with ground truth
iou_ndwi = np.sum(flood_mask & ndwi_water) / np.sum(flood_mask | ndwi_water)
print(f"NDWI IoU: {iou_ndwi:.3f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(flood_mask, cmap='blues')
axes[0].set_title('Ground Truth')
axes[1].imshow(ndwi, cmap='RdYlGn')
axes[1].set_title('NDWI Values')
axes[2].imshow(ndwi_water, cmap='blues')
axes[2].set_title(f'NDWI Water (IoU={iou_ndwi:.3f})')
plt.tight_layout()
plt.show()

## 4. Risk Model Validation

In [ ]:
from app.processing.risk_model import FloodRiskModel

risk_model = FloodRiskModel()

# Test with Jakarta bounding box
jakarta_bbox = (106.5, -6.5, 107.0, -6.0)

risk_result = risk_model.predict_risk(
    bbox=jakarta_bbox,
    rainfall_mm=150,  # Heavy rainfall
    flood_extent=flood_mask
)

print("Risk Assessment Results:")
print(f"  Mean risk: {risk_result['risk_statistics']['mean_risk']:.3f}")
print(f"  High risk area: {risk_result['risk_statistics']['high_risk_area_pct']:.1f}%")
print(f"  Moderate risk area: {risk_result['risk_statistics']['moderate_risk_area_pct']:.1f}%")
print(f"  Low risk area: {risk_result['risk_statistics']['low_risk_area_pct']:.1f}%")
print("\nRecommendations:")
for rec in risk_result['recommendations']:
    print(f"  - {rec}")

## 5. Report Generation Test

In [ ]:
from app.visualization.reporter import ReportGenerator
from app.utils.llm import LLMPromptHandler

# Generate test report
reporter = ReportGenerator()
llm = LLMPromptHandler()

report_data = {
    'location': 'Jakarta, Indonesia',
    'date_range': '2024-01-15 to 2024-01-22',
    'flood_area_km2': 45.5,
    'affected_buildings': 2500,
    'affected_roads_km': 120.5,
    'agricultural_km2': 15.2,
    'rainfall_mm': 185.0,
    'risk_assessment': risk_result['risk_statistics'],
    'recommendations': risk_result['recommendations'],
    'narrative': llm.generate_report(
        location='Jakarta, Indonesia',
        date_range='2024-01-15 to 2024-01-22',
        flood_area_km2=45.5,
        affected_infrastructure={
            'buildings': 2500,
            'roads_km': 120.5,
            'agricultural_km2': 15.2
        }
    )
}

report_path = reporter.generate_report(report_data, 'validation_test')
print(f"Report generated: {report_path}")

## 6. Summary Statistics

In [ ]:
print("\n" + "="*50)
print("VALIDATION SUMMARY")
print("="*50)

print(f"\n✓ SAR Processing: Otsu threshold = {threshold:.2f} dB")
print(f"✓ SAR-IoU: {iou:.3f} (acceptable if > 0.5)")
print(f"✓ NDWI-IoU: {iou_ndwi:.3f} (acceptable if > 0.6)")
print(f"✓ Risk Model: {risk_result['risk_statistics']['mean_risk']:.3f} mean risk")
print(f"✓ Report Generation: {report_path}")

overall_pass = iou > 0.5 and iou_ndwi > 0.6
print(f"\nOverall Validation: {'✓ PASS' if overall_pass else '✗ NEEDS IMPROVEMENT'}")